# Install & Imports

In [1]:
!pip install -q scikit-learn

In [2]:
import os
import glob
import json
from datetime import datetime

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

from sklearn.metrics import classification_report, confusion_matrix

In [3]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


# Paths, Seed, and Dataset Discovery

In [4]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [5]:
BASE_PREPROCESSED_DIR = "/users/"
BASE_MODELS_DIR = "/users/"
DP_FED_DIR = os.path.join(BASE_MODELS_DIR, "DP_Fed")

os.makedirs(DP_FED_DIR, exist_ok=True)
print("📁 DP_FED_DIR:", DP_FED_DIR)

📁 DP_FED_DIR: /users/


In [6]:
def auto_find_dataset(folder_prefix: str) -> str:
    matches = glob.glob(f"{BASE_PREPROCESSED_DIR}/{folder_prefix}*")
    if len(matches) == 0:
        raise ValueError(f"No folder found for prefix: {folder_prefix}")
    if len(matches) > 1:
        print(f"⚠️ Multiple matches for prefix {folder_prefix}, using first:", matches)
    return matches[0]

In [7]:
DATASET_PATHS = {
    "Combined":             auto_find_dataset("Combined"),
}

print("📂 DATASET_PATHS:")
for k, v in DATASET_PATHS.items():
    print(f"  {k}: {v}")

📂 DATASET_PATHS:
  Combined: /users/


# Dataset Loader

In [8]:
def load_preprocessed_dataset(ds_name: str):
    path = DATASET_PATHS[ds_name]

    latent_files = {
        "X_train": "train_latent.npy",
        "X_val":   "val_latent.npy",
        "X_test":  "test_latent.npy",
        "y_train": "y_train.npy",
        "y_val":   "y_val.npy",
        "y_test":  "y_test.npy",
    }

    if ds_name == "Combined":
        latent_files = {
            "X_train": "combined_train_latent.npy",
            "X_val":   "combined_val_latent.npy",
            "X_test":  "combined_test_latent.npy",
            "y_train": "combined_y_train.npy",
            "y_val":   "combined_y_val.npy",
            "y_test":  "combined_y_test.npy",
        }

    X_train = np.load(os.path.join(path, latent_files["X_train"]))
    X_val   = np.load(os.path.join(path, latent_files["X_val"]))
    X_test  = np.load(os.path.join(path, latent_files["X_test"]))

    y_train = np.load(os.path.join(path, latent_files["y_train"]))
    y_val   = np.load(os.path.join(path, latent_files["y_val"]))
    y_test  = np.load(os.path.join(path, latent_files["y_test"]))

    num_classes = len(np.unique(y_train))

    print(f"""📂 Loaded dataset: {ds_name}
  Path: {path}
  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}
  y_train: {y_train.shape}, y_val: {y_val.shape}, y_test: {y_test.shape}
  Num classes: {num_classes}
""")

    return X_train, y_train, X_val, y_val, X_test, y_test, num_classes

# Federated Client Splitter

In [9]:
def make_federated_clients(
    X_train: np.ndarray,
    y_train: np.ndarray,
    num_clients: int = 10,
    shuffle: bool = True,
):
    n_samples = len(X_train)
    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.default_rng(SEED)
        rng.shuffle(indices)

    X_train = X_train[indices]
    y_train = y_train[indices]

    base_size = n_samples // num_clients
    remainder = n_samples % num_clients

    client_splits = []
    start = 0
    for i in range(num_clients):
        size = base_size + (1 if i < remainder else 0)
        end = start + size
        X_c = X_train[start:end]
        y_c = y_train[start:end]
        client_splits.append((X_c, y_c))
        print(f"  -> Client {i}: {len(X_c)} samples")
        start = end

    return client_splits

# FALCON-ID Local Model Definition

In [11]:
def build_falcon_id_local_model(
    input_dim,
    num_classes,
    learning_rate=1e-3,
    l2_reg=1e-4,
    dropout_rate=0.4,
):
    inp = layers.Input(shape=(input_dim,), name="latent_input")
    x = layers.Reshape((input_dim, 1))(inp)  # (batch, 64, 1)

    # ---- Residual Block 1 ----
    shortcut = layers.Conv1D(
        64, 1, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    x1 = layers.Conv1D(
        64, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Activation("relu")(x1)
    x1 = layers.Conv1D(
        64, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x1)
    x1 = layers.BatchNormalization()(x1)

    x = layers.Add()([shortcut, x1])
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # ---- Residual Block 2 ----
    shortcut2 = layers.Conv1D(
        128, 1, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    x2 = layers.Conv1D(
        128, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.Activation("relu")(x2)
    x2 = layers.Conv1D(
        128, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x2)
    x2 = layers.BatchNormalization()(x2)

    x = layers.Add()([shortcut2, x2])
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.SpatialDropout1D(0.2)(x)

    # ---- BiLSTM ----
    x = layers.Bidirectional(
        layers.LSTM(
            128,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(l2_reg),
        )
    )(x)

    # ---- Additive Attention ----
    score = layers.Dense(128, activation="tanh")(x)
    attn_weights = layers.Dense(1, activation="softmax")(score)
    context = layers.Lambda(lambda z: tf.reduce_sum(z[0] * z[1], axis=1))(
        [x, attn_weights]
    )

    # ---- Classifier ----
    x = layers.Dense(
        256, activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(context)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    out = layers.Dense(num_classes, activation="softmax", name="logits")(x)

    model = models.Model(inputs=inp, outputs=out, name="FALCON_ID_CNN_BiLSTM_Attn")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


tmp_X_train, tmp_y_train, _, _, _, _, tmp_num_classes = load_preprocessed_dataset("Combined")
tmp_model = build_falcon_id_local_model(input_dim=tmp_X_train.shape[1], num_classes=tmp_num_classes)
tmp_model.summary()
del tmp_X_train, tmp_y_train, tmp_model

📂 Loaded dataset: Combined
  Path: /users/
  X_train: (9136746, 64), X_val: (1957874, 64), X_test: (1957876, 64)
  y_train: (9136746,), y_val: (1957874,), y_test: (1957876,)
  Num classes: 37



Model: "FALCON_ID_CNN_BiLSTM_Attn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ latent_input        │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 64, 1)     │          0 │ latent_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 64, 64)    │        256 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64)    │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 64, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 64, 64)    │        128 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 64)    │          0 │ conv1d[0][0],     │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 64)    │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 32, 64)    │          0 │ activation_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 32, 128)   │     24,704 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 128)   │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 128)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 32, 128)   │     49,280 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 32, 128)   │      8,320 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 128)   │        512 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 128)   │          0 │ conv1d_3[0][0],   │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 128)   │          0 │ add_1[0][0]       │
│ (Activation)        │                   │            │                 

 Total params: 497,766 (1.90 MB)

 Trainable params: 496,230 (1.89 MB)

 Non-trainable params: 1,536 (6.00 KB)

# DP-SGD Local Training Step (Medium DP)

In [12]:
def dp_train_one_client(
    model: tf.keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int = 512,
    epochs: int = 1,
    l2_clip: float = 1.0,
    noise_multiplier: float = 0.7,
):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    dataset = dataset.shuffle(buffer_size=len(X), seed=SEED).batch(batch_size)

    optimizer = model.optimizer  # use model's optimizer
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

    for _ in range(epochs):
        for xb, yb in dataset:
            with tf.GradientTape() as tape:
                logits = model(xb, training=True)
                loss_val = loss_fn(yb, logits)

            grads = tape.gradient(loss_val, model.trainable_variables)

            # Clip gradients (global L2 norm)
            clipped_grads = []
            for g in grads:
                if g is None:
                    clipped_grads.append(None)
                    continue
                g_norm = tf.norm(g)
                clip_coef = l2_clip / (g_norm + 1e-6)
                clip_coef = tf.minimum(1.0, clip_coef)
                g_clipped = g * clip_coef

                # Add Gaussian noise
                noise = tf.random.normal(
                    shape=tf.shape(g_clipped),
                    stddev=noise_multiplier * l2_clip,
                    seed=SEED,
                )
                g_noisy = g_clipped + noise
                clipped_grads.append(g_noisy)

            optimizer.apply_gradients(zip(clipped_grads, model.trainable_variables))

# Simulated Secure Aggregation

In [13]:
def get_model_weights(model: tf.keras.Model):
    return [w.copy() for w in model.get_weights()]

def set_model_weights(model: tf.keras.Model, weights_list):
    model.set_weights(weights_list)


def secure_aggregate_weights(client_weights_list):
    num_clients = len(client_weights_list)
    num_layers  = len(client_weights_list[0])

    rng = np.random.default_rng(SEED)

    # Generate masks
    masks = []
    for i in range(num_clients):
        layer_masks = []
        for l in range(num_layers):
            shape = client_weights_list[0][l].shape
            m = rng.normal(loc=0.0, scale=1e-3, size=shape)
            layer_masks.append(m)
        masks.append(layer_masks)

    # Last mask ensures sum-to-zero
    for l in range(num_layers):
        total = np.zeros_like(client_weights_list[0][l])
        for i in range(num_clients - 1):
            total += masks[i][l]
        masks[-1][l] = -total

    # Mask and aggregate
    aggregated = []
    for l in range(num_layers):
        layer_sum = np.zeros_like(client_weights_list[0][l])
        for i in range(num_clients):
            layer_sum += client_weights_list[i][l] + masks[i][l]
        aggregated.append(layer_sum / num_clients)

    return aggregated

# DP-FedAvg Training Loop for One Dataset

In [14]:
def run_dp_fedavg_for_dataset(
    ds_name: str,
    num_clients: int = 10,
    num_rounds: int = 5,
    local_epochs: int = 1,
    batch_size: int = 512,
    l2_clip: float = 1.0,
    noise_multiplier: float = 0.7,
    max_train_samples_per_client: int = None,  # Optional speed control
):
    print("\n" + "#" * 80)
    print(f"### 🔐 DP-FedAvg + Secure Aggregation — Dataset: {ds_name}")
    print("#" * 80)

    # 1) Load dataset
    X_train, y_train, X_val, y_val, X_test, y_test, num_classes = load_preprocessed_dataset(ds_name)
    input_dim = X_train.shape[1]

    # 2) Create federated clients
    print(f"👥 Creating {num_clients} federated clients for {ds_name}...")
    client_splits = make_federated_clients(X_train, y_train, num_clients=num_clients)

    # Optional: Subsample per client to reduce DP overhead
    if max_train_samples_per_client is not None:
        new_splits = []
        for i, (X_c, y_c) in enumerate(client_splits):
            if len(X_c) > max_train_samples_per_client:
                X_c = X_c[:max_train_samples_per_client]
                y_c = y_c[:max_train_samples_per_client]
            new_splits.append((X_c, y_c))
            print(f"   Client {i} -> using {len(X_c)} samples (after DP cap)")
        client_splits = new_splits

    # 3) Initialize global model
    global_model = build_falcon_id_local_model(input_dim=input_dim, num_classes=num_classes)

    # For logging
    history = {
        "round": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    # 4) Federated Rounds
    for r in range(1, num_rounds + 1):
        print(f"\n===== 🔁 DP-FedAvg Round {r}/{num_rounds} — Dataset: {ds_name} =====")

        global_weights = get_model_weights(global_model)

        client_weights_after_training = []

        # ----- Local DP training on each client -----
        for cid, (X_c, y_c) in enumerate(client_splits):
            print(f"   🧩 Client {cid}: local DP-SGD training on {len(X_c)} samples...")

            local_model = build_falcon_id_local_model(input_dim=input_dim, num_classes=num_classes)
            set_model_weights(local_model, global_weights)

            dp_train_one_client(
                local_model,
                X_c,
                y_c,
                batch_size=batch_size,
                epochs=local_epochs,
                l2_clip=l2_clip,
                noise_multiplier=noise_multiplier,
            )

            client_weights_after_training.append(get_model_weights(local_model))
            tf.keras.backend.clear_session()

        # ----- Secure Aggregation -----
        print("   🛡️ Secure aggregation of client updates...")
        new_global_weights = secure_aggregate_weights(client_weights_after_training)
        set_model_weights(global_model, new_global_weights)

        # ----- Evaluate global model -----
        train_loss, train_acc = global_model.evaluate(X_train, y_train, verbose=0)
        val_loss, val_acc     = global_model.evaluate(X_val, y_val, verbose=0)

        print(f"   -> Train | loss: {train_loss:.4f}, acc: {train_acc:.4f}")
        print(f"   -> Val   | loss: {val_loss:.4f}, acc: {val_acc:.4f}")

        history["round"].append(r)
        history["train_loss"].append(float(train_loss))
        history["train_acc"].append(float(train_acc))
        history["val_loss"].append(float(val_loss))
        history["val_acc"].append(float(val_acc))

    # 5) Final Test Evaluation
    test_loss, test_acc = global_model.evaluate(X_test, y_test, verbose=0)
    print(f"\n🎯 Final Test Performance on {ds_name} (DP-FedAvg):")
    print(f"   Test loss: {test_loss:.4f}, Test acc: {test_acc:.4f}")

    # 6) Class-wise metrics
    y_pred_prob = global_model.predict(X_test, batch_size=4096, verbose=1)
    y_pred      = np.argmax(y_pred_prob, axis=1)

    cls_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    conf_mat   = confusion_matrix(y_test, y_pred)

    return global_model, history, {
        "test_loss": float(test_loss),
        "test_acc": float(test_acc),
        "classification_report": cls_report,
        "confusion_matrix": conf_mat.tolist(),
        "num_classes": int(num_classes),
    }

# Helpers to Save History & Metrics

In [15]:
import pandas as pd

def save_dp_results(ds_name: str, model, history: dict, metrics: dict):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{ds_name}_DPFedAvg_{timestamp}"

    # 1) Save model
    model_path = os.path.join(DP_FED_DIR, f"{prefix}_model.keras")
    model.save(model_path)
    print(f"💾 Saved model at: {model_path}")

    # 2) Save history as CSV
    hist_df = pd.DataFrame(history)
    hist_csv_path = os.path.join(DP_FED_DIR, f"{prefix}_history.csv")
    hist_df.to_csv(hist_csv_path, index=False)
    print(f"💾 Saved training history at: {hist_csv_path}")

    # 3) Save metrics as JSON
    metrics_path = os.path.join(DP_FED_DIR, f"{prefix}_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"💾 Saved metrics at: {metrics_path}")

# Run DP-FedAvg on All Datasets (Medium DP)

In [16]:
NUM_CLIENTS = 10
NUM_ROUNDS  = 3
LOCAL_EPOCHS = 1
BATCH_SIZE   = 4096

L2_CLIP = 1.0
NOISE_MULTIPLIER = 0.7
MAX_TRAIN_SAMPLES_PER_CLIENT = 100000

global_summary_rows = []

for ds_name in DATASET_PATHS.keys():
    model_ds, history_ds, metrics_ds = run_dp_fedavg_for_dataset(
        ds_name=ds_name,
        num_clients=NUM_CLIENTS,
        num_rounds=NUM_ROUNDS,
        local_epochs=LOCAL_EPOCHS,
        batch_size=BATCH_SIZE,
        l2_clip=L2_CLIP,
        noise_multiplier=NOISE_MULTIPLIER,
        max_train_samples_per_client=MAX_TRAIN_SAMPLES_PER_CLIENT,
    )

    save_dp_results(ds_name, model_ds, history_ds, metrics_ds)

    global_summary_rows.append({
        "dataset": ds_name,
        "num_clients": NUM_CLIENTS,
        "num_rounds": NUM_ROUNDS,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "l2_clip": L2_CLIP,
        "noise_multiplier": NOISE_MULTIPLIER,
        "test_loss": metrics_ds["test_loss"],
        "test_acc": metrics_ds["test_acc"],
    })

# Save global summary
global_df = pd.DataFrame(global_summary_rows)
global_summary_path = os.path.join(DP_FED_DIR, "DPFedAvg_summary_all_datasets.csv")
global_df.to_csv(global_summary_path, index=False)

print("\n" + "#" * 80)
print("✅ Completed DP-FedAvg + Secure Aggregation for all datasets.")
print("📊 Global summary saved at:", global_summary_path)
print("#" * 80)
display(global_df)


################################################################################
### 🔐 DP-FedAvg + Secure Aggregation — Dataset: Combined
################################################################################
📂 Loaded dataset: Combined
  Path: /users/
  X_train: (9136746, 64), X_val: (1957874, 64), X_test: (1957876, 64)
  y_train: (9136746,), y_val: (1957874,), y_test: (1957876,)
  Num classes: 37

👥 Creating 10 federated clients for Combined...
  -> Client 0: 913675 samples
  -> Client 1: 913675 samples
  -> Client 2: 913675 samples
  -> Client 3: 913675 samples
  -> Client 4: 913675 samples
  -> Client 5: 913675 samples
  -> Client 6: 913674 samples
  -> Client 7: 913674 samples
  -> Client 8: 913674 samples
  -> Client 9: 913674 samples
   Client 0 -> using 100000 samples (after DP cap)
   Client 1 -> using 100000 samples (after DP cap)
   Client 2 -> using 100000 samples (after DP cap)
   Client 3 -> using 100000 samples (after DP cap)
   Client 4 -> using 100000 samples

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (4096, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (1696, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


   🧩 Client 1: local DP-SGD training on 100000 samples...
   🧩 Client 2: local DP-SGD training on 100000 samples...
   🧩 Client 3: local DP-SGD training on 100000 samples...
   🧩 Client 4: local DP-SGD training on 100000 samples...
   🧩 Client 5: local DP-SGD training on 100000 samples...
   🧩 Client 6: local DP-SGD training on 100000 samples...
   🧩 Client 7: local DP-SGD training on 100000 samples...
   🧩 Client 8: local DP-SGD training on 100000 samples...
   🧩 Client 9: local DP-SGD training on 100000 samples...
   🛡️ Secure aggregation of client updates...


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


   -> Train | loss: 3.8340, acc: 0.0037
   -> Val   | loss: 3.8337, acc: 0.0037

===== 🔁 DP-FedAvg Round 2/3 — Dataset: Combined =====
   🧩 Client 0: local DP-SGD training on 100000 samples...
   🧩 Client 1: local DP-SGD training on 100000 samples...
   🧩 Client 2: local DP-SGD training on 100000 samples...
   🧩 Client 3: local DP-SGD training on 100000 samples...
   🧩 Client 4: local DP-SGD training on 100000 samples...
   🧩 Client 5: local DP-SGD training on 100000 samples...
   🧩 Client 6: local DP-SGD training on 100000 samples...
   🧩 Client 7: local DP-SGD training on 100000 samples...
   🧩 Client 8: local DP-SGD training on 100000 samples...
   🧩 Client 9: local DP-SGD training on 100000 samples...
   🛡️ Secure aggregation of client updates...
   -> Train | loss: 3.9481, acc: 0.0453
   -> Val   | loss: 3.9478, acc: 0.0454

===== 🔁 DP-FedAvg Round 3/3 — Dataset: Combined =====
   🧩 Client 0: local DP-SGD training on 100000 samples...
   🧩 Client 1: local DP-SGD training on 100000

,dataset,num_clients,num_rounds,local_epochs,batch_size,l2_clip,noise_multiplier,test_loss,test_acc
0,Combined,10,3,1,4096,1.0,0.7,4.10643,0.061059
